# Gift-Eval Ablation Analysis

This notebook analyzes the effects of a TSFM's performance across different datasets.

**Ablation Levels:**
- Level 0: Original model (no ablation)
- Level 1: Ablate layers 10, heads-all
- Level 2: Ablate layers 10-11, heads-all
- Level 3: Ablate layers 10-11-12, heads-all
- Level 4: Ablate layers 10-11-12-13, heads-all
- Level 5: Ablate layers 10-11-12-13-14, heads-all
- Level 6: Ablate layers 10-11-12-13-14-15, heads-all


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from tsfm_lens.utils import apply_custom_style, normalize_by_seasonal_naive
from scipy.stats import gmean
from collections import defaultdict

In [ ]:
apply_custom_style("../../config/plotting.yaml")

In [ ]:
HOME_DIR = os.getenv("HOME")

In [ ]:
root_dir = os.path.join(HOME_DIR, "tsfm-lens")

In [ ]:
model_to_dir_mapping = {
    "TimesFM 2.5": "google-timesfm-2.5-200m-pytorch",
    "Chronos Bolt": "amazon-chronos-bolt-base",
    "Chronos": "amazon-chronos-t5-base",
    "Toto": "Datadog-Toto-Open-Base-1.0_samples-20",
}

In [ ]:
# Select the metric to analyze
SELECTED_METRIC = "eval_metrics/MASE[0.5]"
# SELECTED_METRIC = "eval_metrics/sMAPE[0.5]"

metric_pretty_name = "".join(c for c in SELECTED_METRIC.split("/")[-1] if c.isalpha())

print(f"Analyzing metric: {SELECTED_METRIC}")
print(f"Metric name: {metric_pretty_name}")


In [ ]:
save_figs = True
figs_save_dir = os.path.join(root_dir, "figs", f"gift-eval_{metric_pretty_name}", "combined")
if not os.path.exists(figs_save_dir):
    os.makedirs(figs_save_dir)
if save_figs:
    print(f"Saving figures to: {figs_save_dir}")

## Load Data


In [ ]:
# Heads and MLP ablation
head_mlp_ablation_configs = {
    "TimesFM 2.5": {
        "selected_layers_lst": [
            11,
            [10, 11],
            [10, 11, 12],
            [10, 11, 12, 13],
            [9, 10, 11, 12, 13],
            [7, 8, 9, 10, 11, 12],
            [7, 8, 9, 10, 11, 12, 13],
        ],
        "n_divisor": 20,
    },
    "Chronos Bolt": {
        "selected_layers_lst": [2, [1, 2], [1, 2, 3], [1, 2, 3, 4], [1, 2, 3, 4, 5], [1, 2, 3, 4, 5, 6]],
        "n_divisor": 12,
    },
    "Chronos": {
        "selected_layers_lst": [],
        "n_divisor": 12,
    },
    "Toto": {
        "selected_layers_lst": [10, [9, 10], [3, 4, 5], [5, 6, 7, 8], [4, 5, 6, 7, 8]],
        "n_divisor": 12,
    },
}

# Heads ablation
head_ablation_configs = {
    "TimesFM 2.5": {
        "selected_layers_lst": [
            10,
            [10, 11],
            [10, 11, 12],
            [10, 11, 12, 13],
            [7, 8, 9, 10, 11],
            [7, 8, 9, 10, 11, 12],
        ],
        "n_divisor": 20,
    },
    "Chronos Bolt": {
        "selected_layers_lst": [2, [1, 2], [1, 2, 3], [1, 2, 3, 4], [1, 2, 3, 4, 5], [1, 2, 3, 4, 5, 6]],
        "n_divisor": 12,
    },
    "Chronos": {
        "selected_layers_lst": [[7, 8], [5, 6, 7], [5, 6, 7, 8]],
        "n_divisor": 12,
    },
    "Toto": {
        "selected_layers_lst": [10, [9, 10], [9, 10, 11], [2, 8, 9, 10], [7, 8, 9, 10, 11]],
        "n_divisor": 12,
    },
}

# MLP ablation
mlp_ablation_configs = {
    "TimesFM 2.5": {
        "selected_layers_lst": [
            9,
            [8, 9],
            [7, 8, 9],
            [6, 7, 8, 9],
            [8, 9, 10, 11, 12],
            [8, 9, 10, 11, 12, 13],
            [7, 8, 9, 10, 11, 12, 13],
        ],
        "n_divisor": 20,
    },
    "Chronos Bolt": {
        "selected_layers_lst": [8, [2, 3], [3, 4, 5], [3, 4, 5, 6], [2, 3, 4, 5, 6], [3, 4, 5, 6, 7, 8]],
        "n_divisor": 12,
    },
    "Chronos": {
        "selected_layers_lst": [],
        "n_divisor": 12,
    },
    "Toto": {
        "selected_layers_lst": [10, [8, 9], [6, 7, 8], [5, 6, 7, 8], [7, 8, 9, 10, 11]],
        "n_divisor": 12,
    },
}


# Helper function to build ablation file mappings
def build_ablation_files(configs, file_prefix):
    files_by_model = defaultdict(dict)

    for model_name, config in configs.items():
        files_by_model[model_name][0] = "original_all_results.csv"

        for selected_layers in config["selected_layers_lst"]:
            if isinstance(selected_layers, list):
                selected_layers_str = "-".join(map(str, selected_layers))
                num_layers = len(selected_layers)
            else:
                selected_layers_str = str(selected_layers)
                num_layers = 1

            files_by_model[model_name][num_layers] = (
                f"{file_prefix}_layers_{selected_layers_str}_heads-all_all_results.csv"
            )

    return files_by_model


head_mlp_ablation_files_by_model = build_ablation_files(head_mlp_ablation_configs, "head-mlp")
head_ablation_files_by_model = build_ablation_files(head_ablation_configs, "head")
mlp_ablation_files_by_model = build_ablation_files(mlp_ablation_configs, "mlp")


In [ ]:
head_ablation_files_by_model["TimesFM 2.5"]

In [ ]:
rseeds_lst = [42, 123]


# Load and combine CSV files across ablation levels and rseeds
def load_ablation_data(ablated_files_dict, metrics_dir_dict):
    """Load ablation data, selecting best rseed per level based on geometric mean of selected metric."""
    best_filepaths_by_key = {}
    data_by_key = {}

    # Handle case where ablated_files_dict is a simple dict mapping level -> filepath
    if all(isinstance(v, str) for v in ablated_files_dict.values()):
        ablated_files_dict = {"default": ablated_files_dict}

    for key, ablation_files in ablated_files_dict.items():
        data_by_key[key] = {}
        best_filepaths_by_key[key] = {}
        curr_data_frame = {}

        for level, filepath in ablation_files.items():
            rseed_data = {}

            # Load data for each rseed
            for rseed, metrics_dir in metrics_dir_dict.items():
                df_filepath = os.path.join(metrics_dir, filepath)
                if not os.path.exists(df_filepath):
                    print(f"Level {level}, rseed {rseed}: Skipping (not found)")
                    continue

                df = pd.read_csv(df_filepath)
                if len(df) != 97:
                    print(f"Level {level}, rseed {rseed}: Skipping (incomplete)")
                    continue

                print(f"Level {level}, rseed {rseed}: Loading")
                df["ablation_level"] = level
                rseed_data[rseed] = (df, df_filepath)

            if not rseed_data:
                print(f"Level {level}: No data found, skipping")
                continue

            # Select rseed with lowest geometric mean of selected metric
            rseed_gmeans = {}
            for rseed, (df, df_filepath) in rseed_data.items():
                selected_metric_gmean = gmean(df[SELECTED_METRIC])
                rseed_gmeans[rseed] = {
                    f"gmean_{SELECTED_METRIC}": selected_metric_gmean,
                    "filepath": df_filepath,
                    "df": df,
                }

            best_rseed = min(rseed_gmeans.keys(), key=lambda r: rseed_gmeans[r][f"gmean_{SELECTED_METRIC}"])
            best_df = rseed_gmeans[best_rseed]["df"]
            best_filepath = rseed_gmeans[best_rseed]["filepath"]

            # Store best filepath information
            best_filepaths_by_key[key][level] = {
                "best_rseed": best_rseed,
                "best_filepath": best_filepath,
                "available_rseeds": list(rseed_data.keys()),
                f"gmean_{SELECTED_METRIC}": rseed_gmeans[best_rseed][f"gmean_{SELECTED_METRIC}"],
                "num_rseeds_evaluated": len(rseed_data),
            }

            curr_data_frame[level] = best_df
            print(
                f"Level {level}: Best rseed={best_rseed} (gmean={rseed_gmeans[best_rseed][f'gmean_{SELECTED_METRIC}']:.6f}) from {len(rseed_data)} rseeds"
            )

        # Only concatenate if we have data
        if curr_data_frame:
            all_data_for_key = pd.concat(curr_data_frame.values(), ignore_index=True)
            data_by_key[key] = all_data_for_key
            print(f"\nTotal: {len(all_data_for_key)} rows, {all_data_for_key['dataset'].nunique()} unique datasets")
        else:
            print(f"\nNo data found for key {key}")
            data_by_key[key] = pd.DataFrame()  # Empty dataframe

    # If we only had one key (default), return the data directly instead of nested
    if len(data_by_key) == 1 and "default" in data_by_key:
        return best_filepaths_by_key["default"], data_by_key["default"]

    return best_filepaths_by_key, data_by_key


# Load data for all models and ablation types
all_ablation_data = {}

for model_name in head_mlp_ablation_configs.keys():
    print(f"\n{'=' * 80}")
    print(f"Loading data for model: {model_name}")
    print(f"{'=' * 80}")

    # Create metrics directory dict for this model
    metrics_dir_dict = {
        rseed: os.path.join(root_dir, "results", model_to_dir_mapping[model_name], f"all_components_rseed-{rseed}")
        for rseed in rseeds_lst
    }
    print(f"Metrics directory: {metrics_dir_dict}")

    # Load data for each ablation type
    print(f"\n--- Loading head+mlp ablation data for {model_name} ---")
    head_mlp_data_frames, head_mlp_all_data = load_ablation_data(
        head_mlp_ablation_files_by_model[model_name], metrics_dir_dict
    )

    print(f"\n--- Loading head ablation data for {model_name} ---")
    head_data_frames, head_all_data = load_ablation_data(head_ablation_files_by_model[model_name], metrics_dir_dict)

    print(f"\n--- Loading mlp ablation data for {model_name} ---")
    mlp_data_frames, mlp_all_data = load_ablation_data(mlp_ablation_files_by_model[model_name], metrics_dir_dict)

    # Store in consolidated dictionary
    all_ablation_data[model_name] = {
        "head_mlp": {"data_frames": head_mlp_data_frames, "all_data": head_mlp_all_data},
        "head": {"data_frames": head_data_frames, "all_data": head_all_data},
        "mlp": {"data_frames": mlp_data_frames, "all_data": mlp_all_data},
    }

print(f"\n{'=' * 80}")
print(f"Data loading complete for all models")
print(f"{'=' * 80}")

## Optional: Normalize by Seasonal Naive Baseline


In [ ]:
# Configuration: Set to True to normalize metrics by seasonal naive baseline
NORMALIZE_BY_SEASONAL_NAIVE = True

print(f"Normalization by seasonal naive baseline: {NORMALIZE_BY_SEASONAL_NAIVE}")


## 1. Box Plot: Overall Performance Across Ablation Levels


In [ ]:
model_name = "Chronos Bolt"

# Prepare data for box plot
selected_head_mlp_box_data = all_ablation_data[model_name]["head_mlp"]["all_data"][
    ["dataset", "ablation_level", SELECTED_METRIC]
].copy()
selected_mlp_box_data = all_ablation_data[model_name]["mlp"]["all_data"][
    ["dataset", "ablation_level", SELECTED_METRIC]
].copy()
selected_head_all_box_data = all_ablation_data[model_name]["head"]["all_data"][
    ["dataset", "ablation_level", SELECTED_METRIC]
].copy()

# Remove any infinite or NaN values
selected_head_mlp_box_data = selected_head_mlp_box_data.replace([np.inf, -np.inf], np.nan).dropna()
selected_mlp_box_data = selected_mlp_box_data.replace([np.inf, -np.inf], np.nan).dropna()
selected_head_all_box_data = selected_head_all_box_data.replace([np.inf, -np.inf], np.nan).dropna()

In [ ]:
if NORMALIZE_BY_SEASONAL_NAIVE:
    print("\nLoading seasonal naive baseline...")
    seasonal_naive_path = os.path.join(root_dir, "results", "seasonal_naive_baseline", "all_results.csv")
    seasonal_naive_df = pd.read_csv(seasonal_naive_path)
    print(f"Loaded seasonal naive baseline: {len(seasonal_naive_df)} datasets")

    print("\nNormalizing data by seasonal naive baseline...")
    selected_head_mlp_box_data = normalize_by_seasonal_naive(selected_head_mlp_box_data, seasonal_naive_df)
    selected_mlp_box_data = normalize_by_seasonal_naive(selected_mlp_box_data, seasonal_naive_df)
    selected_head_all_box_data = normalize_by_seasonal_naive(selected_head_all_box_data, seasonal_naive_df)
else:
    print("\nSkipping normalization.")

## 2. Line Plot: Median Performance Trend


In [ ]:
# Calculate median and percentiles for each ablation level
def compute_stats(data, metric):
    return (
        data.groupby("ablation_level")[metric]
        .agg(
            [
                "median",
                "mean",
                lambda x: gmean(x),
                lambda x: np.exp(np.std(np.log(x)) / np.sqrt(len(x))),  # geometric standard error
                lambda x: x.quantile(0.25),
                lambda x: x.quantile(0.75),
            ]
        )
        .rename(columns={"<lambda_0>": "geom_mean", "<lambda_1>": "geom_ste", "<lambda_2>": "q25", "<lambda_3>": "q75"})
        .reset_index()
    )


head_mlp_stats_by_level = compute_stats(selected_head_mlp_box_data, SELECTED_METRIC)
mlp_stats_by_level = compute_stats(selected_mlp_box_data, SELECTED_METRIC)
head_all_stats_by_level = compute_stats(selected_head_all_box_data, SELECTED_METRIC)


In [ ]:
# Plot each ablation type
# colors = plt.cm.get_cmap("Set1")(range(3))
# get default color cycle
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
print(colors)
plot_configs = [
    (head_all_stats_by_level, "Ablate Heads", colors[0], "^", 7, "-"),
    (mlp_stats_by_level, "Ablate MLPs", colors[1], "s", 7, "-"),
    (head_mlp_stats_by_level, "Ablate Heads + MLP", colors[2], "p", 9, "-"),
]

In [ ]:
# Create line plot with error bands
fig, ax = plt.subplots(figsize=(3.5, 5))

for stats, label, color, marker, markersize, linestyle in plot_configs:
    ax.plot(
        stats["ablation_level"],
        stats["geom_mean"],
        marker=marker,
        linewidth=2.5,
        markersize=markersize,
        # markerfacecolor="None",
        markeredgecolor=color,
        markeredgewidth=2,
        label=label,
        color=color,
        linestyle=linestyle,
        alpha=1,
        zorder=3,
    )
    ax.fill_between(
        stats["ablation_level"],
        stats["geom_mean"] * (stats["geom_ste"] ** 0.33),
        stats["geom_mean"] / (stats["geom_ste"] ** 0.33),
        alpha=0.1,
        color=color,
        zorder=1,
    )

baseline_geom_mean = stats.loc[stats["ablation_level"] == 0, "geom_mean"].values[0]
# plot dashed line for baseline
ax.axhline(
    y=baseline_geom_mean,
    color="red",
    linestyle="--",
    linewidth=2,
    # label=f"Baseline ({baseline_geom_mean:.3f})",
    alpha=0.7,
    zorder=4,
)
# Add annotation at y-axis intersection
ax.text(
    0,
    baseline_geom_mean * 0.994,
    f"Baseline ({baseline_geom_mean:.3f})",
    color="red",
    fontsize=8,
    fontweight="bold",
    verticalalignment="top",
    horizontalalignment="left",
    zorder=100,
)
# plot dashed line for 1.002 * baseline_geom_mean
ax.axhline(
    y=1.02 * baseline_geom_mean,
    color="orangered",
    linestyle="--",
    linewidth=2,
)
# Add annotation at y-axis intersection
ax.text(
    0,
    (1.02 * baseline_geom_mean) * 1.004,
    f"Baseline + 2%",
    color="orangered",
    fontsize=8,
    fontweight="bold",
    verticalalignment="bottom",
    horizontalalignment="left",
    zorder=100,
)


# Formatting
ax.set_xlabel("Number of Contiguous Layers Ablated", fontweight="bold", fontsize=9)
ax.set_ylabel(f"{metric_pretty_name} (Geom Mean)", fontweight="bold", fontsize=9)
# ax.set_title(f"ZA All Components in Contiguous Layers ({model_name})", fontweight="bold", pad=20)
ax.set_title(model_name, fontweight="bold", pad=10)
ax.grid(True, alpha=0.2)
ax.legend(loc="lower right", frameon=True, fontsize=8)

# Combine xticks from all datasets
all_levels = sorted(
    set(head_mlp_stats_by_level["ablation_level"].unique())
    | set(mlp_stats_by_level["ablation_level"].unique())
    | set(head_all_stats_by_level["ablation_level"].unique())
)
xticks = all_levels
ax.set_xticks(xticks)
ax.set_xticklabels([f"{int(x)}" if int(x) == x else f"{x:.1f}" for x in xticks], rotation=0)

ylow = 0.96 * baseline_geom_mean
yhigh = 1.12 * baseline_geom_mean
ax.set_ylim(ylow, yhigh)

plt.tight_layout()
save_path = os.path.join(figs_save_dir, f"comparison_line_plot_{model_name}.pdf")
if save_figs:
    plt.savefig(save_path, bbox_inches="tight")
    print(f"Saved comparison line plot to: {save_path}")
plt.show()

In [ ]:
rseeds_lst

In [ ]:
# Bar and Dynamite Plot: Level 0 vs Level 1 ablations (heads, mlps, heads+mlps)
# Load seasonal naive baseline for normalization
seasonal_naive_path = os.path.join(root_dir, "results", "seasonal_naive_baseline", "all_results.csv")
seasonal_naive_df = pd.read_csv(seasonal_naive_path)
print(f"Loaded seasonal naive baseline: {len(seasonal_naive_df)} datasets")

ablation_types = {"head": "Ablate Heads", "mlp": "Ablate MLPs", "head_mlp": "Ablate Entire Layer"}


def compute_bar_stats(df, level):
    """Extract stats for a given ablation level from normalized data."""
    if df.empty:
        return None
    level_data = df[df["ablation_level"] == level]
    if level_data.empty:
        return None
    values = level_data[SELECTED_METRIC].values
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return None
    return {
        "geom_mean": gmean(values),
        "geom_ste": np.exp(np.std(np.log(values)) / np.sqrt(len(values))),
        "n": len(values),
    }


# Build bar plot data
bar_plot_data = []
for model_name, model_data in all_ablation_data.items():
    # Normalize all ablation types
    normalized = {
        k: normalize_by_seasonal_naive(model_data[k]["all_data"].copy(), seasonal_naive_df)
        if not model_data[k]["all_data"].empty
        else pd.DataFrame()
        for k in ablation_types
    }

    # Add original (level 0) from head ablation
    if stats := compute_bar_stats(normalized["head"], level=0):
        bar_plot_data.append({"model": model_name, "condition": "Original", **stats})

    # Add level 1 ablations
    for abl_type, condition in ablation_types.items():
        if stats := compute_bar_stats(normalized[abl_type], level=1):
            bar_plot_data.append({"model": model_name, "condition": condition, **stats})

bar_df = pd.DataFrame(bar_plot_data)
print("\nUsing data normalized by seasonal naive baseline:")
print(bar_df)


In [ ]:
# Create horizontal bar and dynamite plot
models = ["TimesFM 2.5", "Chronos Bolt", "Toto", "Moirai"][::-1]  # Reverse for top-to-bottom order
conditions = ["Original", "Ablate MLPs", "Ablate Heads", "Ablate Entire Layer"]
labels = {
    "Original": "Original",
    "Ablate MLPs": "Ablate MLP",
    "Ablate Heads": "Ablate Heads",
    "Ablate Entire Layer": "Ablate Heads & MLP",
}
default_colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
condition_colors = ["lightgray", default_colors[1], default_colors[0], default_colors[2]]

fig, ax = plt.subplots(figsize=(6, 6))
bar_height = 0.22
y = np.arange(len(models))

# Pivot data for efficient lookup
pivot_mean = bar_df.pivot(index="model", columns="condition", values="geom_mean").reindex(models)
pivot_ste = bar_df.pivot(index="model", columns="condition", values="geom_ste").reindex(models)

for i, condition in enumerate(conditions):
    means = pivot_mean[condition].values
    stes = pivot_ste[condition].values
    xerr = [means - means / stes, means * stes - means]  # Asymmetric error bars

    ax.barh(
        y + i * bar_height,
        means,
        bar_height,
        label=labels[condition],
        color=condition_colors[i],
        alpha=0.85,
        edgecolor="black",
        linewidth=0.8,
    )
    ax.errorbar(
        means, y + i * bar_height, xerr=xerr, fmt="none", color="darkgray", capsize=3, capthick=1.5, elinewidth=1.5
    )

    # Add percentage annotations for ablation conditions
    if condition != "Original":
        originals = pivot_mean["Original"].values
        pct = np.where(np.abs((means - originals) / originals * 100) < 0.05, 0.0, (means - originals) / originals * 100)
        for j, (m, p) in enumerate(zip(means, pct)):
            if not np.isnan(m):
                ax.text(
                    (0.6 + m) / 2,
                    y[j] + i * bar_height,
                    f"{p:+.1f}%",
                    ha="center",
                    va="center",
                    fontsize=12,
                    fontweight="bold",
                    color="white",
                )

# Formatting
ax.set_xlabel(f"{metric_pretty_name} (Geom Mean)", fontweight="bold", fontsize=12)
ax.set_title("Single Layer Component Ablations", fontweight="bold", fontsize=14, pad=15)
ax.set_yticks(y + bar_height * (len(conditions) - 1) / 2)
ax.set_yticklabels(models, fontsize=11)
ax.legend(loc="upper right", frameon=True, fontsize=10)
ax.grid(True, alpha=0.3, axis="x")
ax.set_xlim(0.6, None)
plt.tight_layout()

if save_figs:
    save_path = os.path.join(figs_save_dir, "ablation_comparison_single_layer.pdf")
    plt.savefig(save_path, bbox_inches="tight")
    print(f"Saved bar plot to: {save_path}")
plt.show()


In [ ]:
# Print summary statistics table
print("Summary of Ablation Effects (Level 2):")
print("=" * 60)
pivot_df = bar_df.pivot(index="model", columns="condition", values="geom_mean")
pivot_df = pivot_df[conditions]  # Reorder columns
print(pivot_df.round(4))
